# Saving and Loading a Machine Learning Pipeline using `pickle`
This notebook demonstrates how to **train, save, and load** a machine learning pipeline using `pickle`. The goal is to reuse the trained model for classification on **new unseen data** without retraining.

---
## Steps in the Process:
### 1️. Train and Save the Model
- Load the dataset.
- Preprocess data:
  - Handle missing values.
  - Scale numerical features.
  - Encode categorical features.
- Train a **Logistic Regression** model inside a `Pipeline`.
- Save the trained pipeline using `pickle`.

### 2️. Save the Dependencies
- Generate a `requirements.txt` file to ensure the same library versions are installed on another machine.
  ```bash
  pip freeze > requirements.txt

- Besides the dependencies, you will also need to save your custom preprocessing functions as .py functions. Please keep the function with the following name `feature_engineering.py` where you should perform all the feature eng. 

#### **Files to submit**
We save **two separate files** to ensure **modularity, flexibility, and compatibility**:

| **File** | **Purpose** | **Why Separate?** |
|----------|------------|------------------|
| `Pipeline.pkl` | Stores **feature engineering, scaling, encoding, transformations and trainig models** | Ensures new data is processed **exactly like training data** and we can apply the best model. |
| `feature_engineering.py` | Stores the feature engineering processing steps | Necessary to run the pipeline once it is not a native scikit-learn | 

### 3. Load the Saved Model and Make Predictions (Please try it in a new notebook)
 - Transfer the saved model (`.pkl` file) and `requirements.txt` to another machine.
 - Install dependencies:`pip install -r requirements.txt`
 - Import the feature engineer function `from feature_engineering import feature_engineering`
 - Load the saved pipeline and classify new, unseen data

**Note**: depending on the algorithms use it might be necessary to save two files, one for processing and another to run the model.

# Example 1 - Scikit-learn based method

#### 1.1 Train and Save the Model

In [4]:
import pandas as pd
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Load dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = pd.read_csv(url, names=names)

# Identify numerical and categorical columns
numerical_features = ['plas', 'pres', 'skin', 'mass', 'pedi', 'age']  # Excluding 'test' column
categorical_features = ['preg']  # Assume 'preg' is categorical for this example

# Convert categorical column(s) to type 'category'
dataframe[categorical_features] = dataframe[categorical_features].astype("category")

# Introduce some NaN values artificially (for testing missing value handling)
dataframe.loc[5:10, "plas"] = np.nan  # Simulating missing values

# Split into input (X) and output (Y) variables
X = dataframe.iloc[:, 0:8]  # Features
Y = dataframe.iloc[:, 8]  # Target variable

# Define custom function to drop a column & create new feature
def feature_engineering(X):
    """Drop 'test' column and create a new interaction feature 'BMI_Age_Interaction'."""
    X = pd.DataFrame(X, columns=numerical_features + categorical_features)
    
    # Drop the 'test' column if it exists
    if 'test' in X.columns:
        X = X.drop(columns=['test'], errors='ignore')

    # Create a new feature: BMI_Age_Interaction = mass * age
    X["BMI_Age_Interaction"] = X["mass"] * X["age"]

    return X

# Apply FunctionTransformer to apply the feature engineering function
feature_transformer = FunctionTransformer(feature_engineering)

## 1️ Numerical Processing: Handle NaN & Apply Z-Normalization
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Replace NaN with median
    ('scaler', StandardScaler())  # Apply Z-normalization
])

## 2️ Categorical Processing: One-Hot Encoding
categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # Convert categorical to numeric
])

# 3 Combine both transformations into a ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

# Create the full pipeline with preprocessing + feature engineering + model
pipeline = Pipeline([
    ('feature_engineering', feature_transformer),  # Drop column & calculate a new feature based on X["mass"] * X["age"]
    ('preprocessor', preprocessor),  # Handle missing values, scale, and encode
    ('model', LogisticRegression(max_iter=500))  # Train logistic regression model
])

# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.33, random_state=7)

# Fit the pipeline
pipeline.fit(X_train, Y_train)

# Save the pipeline using pickle
filename = 'saved_pipeline.pkl' # use what is asked in the assignment for the name
with open(filename, 'wb') as file:
    pickle.dump(pipeline, file)
 
print(f"Pipeline saved as {filename}")


Pipeline saved as saved_pipeline.pkl


#### 1.2 You now should apply the freeze command to get the requirements

#### 1.3 Now install the requirements, load the model and apply it

In [ ]:
import pickle
from feature_engineering import feature_engineering # new

# Load the saved pipeline
filename = 'saved_pipeline.pkl'
with open(filename, 'rb') as file:
    loaded_pipeline = pickle.load(file)

# Example: New unseen data (single instance)
new_data = pd.DataFrame({
    'preg': [3],    # Categorical feature
    'plas': [150],  # Numerical feature
    'pres': [72],
    'skin': [35],
    'test': [0],    # This will be dropped automatically
    'mass': [33.6],
    'pedi': [0.627],
    'age': [50]
})

# Make predictions
prediction = loaded_pipeline.predict(new_data)

print(f"Predicted Class: {prediction[0]}")

Predicted Class: 1


# Example 2 - Method outside scikit learn models (e.x. XGBoost, but can be other)

## Why Do We Need Two Pickle Files When Models Are Outside Scikit-Learn?

When working with **machine learning models that are outside the `scikit-learn` framework** (e.g., XGBoost, LightGBM, TensorFlow, PyTorch), we need to handle **preprocessing and model storage separately**. This is because non-Scikit-learn models do not naturally integrate into `Pipeline` objects, requiring us to manage transformations manually.

---

### **Why Separate Pickle Files?**
We save **two separate pickle files** to ensure **modularity, flexibility, and compatibility**:

| **File** | **Purpose** | **Why Separate?** |
|----------|------------|------------------|
| `preprocessor.pkl` | Stores **feature engineering, scaling, encoding, and transformations** | Ensures new data is processed **exactly like training data** before inference. |
| `xgb_model.pkl` | Stores the **trained machine learning model** | Keeps the model independent from transformations, allowing flexibility to swap models later. |
| `feature_engineering.py` | Stores the feature engineering processing steps | Necessary to run the pipeline once it is not a native scikit-learn | 

---

### **Key Reasons for This Approach**
#### **1️ Preprocessing & Feature Engineering Must Be Consistent**
- The same transformations (e.g., **scaling, encoding, missing value handling**) **must be applied** to both **training and test data**.
- If preprocessing is **not saved separately**, new data may be **processed incorrectly**, leading to **wrong predictions**.

#### **2️ XGBoost (and Other Models) Do Not Integrate with Pipelines**
- XGBoost models **cannot be embedded** inside a `Pipeline` like `scikit-learn` models.
- `scikit-learn` pipelines **only work with compatible models**, like `LogisticRegression`, `RandomForest`, etc.
- Saving XGBoost separately ensures it can be **loaded independently**.

#### **3️ Flexibility: Change Models Without Reprocessing Data**
- If preprocessing is stored separately, we can **train different models (e.g., Random Forest, LightGBM) without reapplying transformations**.
- This **saves time** and **allows experimentation with multiple models** using the same processed data.

#### **4️ Cross-Platform Compatibility**
- `scikit-learn`'s preprocessing steps are designed for structured data.
- `XGBoost` models are optimized for **speed and efficiency**, and saving them separately ensures **better performance and portability**.

---

### **Summary: Best Practice for Saving Models Outside Scikit-Learn**
| **Step** | **Action** | **File Type** |
|----------|-----------|--------------|
| 1️ Save preprocessing & feature engineering | Pickle the pipeline | `preprocessor.pkl` |
| 2️ Train & save the XGBoost model | Pickle the model | `xgb_model.pkl` |
| 3️ Load both files before inference | Apply transformations & predict | `preprocessor.pkl` + `xgb_model.pkl` |

---

## **🚀 Final Thoughts**
- **Preprocessing and feature engineering must be saved separately** so that new data is processed the same way as training data.
- **Machine learning models (like XGBoost) should be stored independently** for flexibility and compatibility.
- **This approach allows you to switch models easily** while keeping preprocessing consistent.

By following this method, we create a **modular, efficient, and reproducible** machine learning workflow! 🎯🚀


#### 2.1 Train and Save the Model

In [1]:
import pandas as pd
import pickle
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Load dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = pd.read_csv(url, names=names)

# Identify numerical and categorical columns
numerical_features = ['plas', 'pres', 'skin', 'mass', 'pedi', 'age']  # Excluding 'test' column
categorical_features = ['preg']  # Assume 'preg' is categorical for this example

# Convert categorical column(s) to type 'category'
dataframe[categorical_features] = dataframe[categorical_features].astype("category")

# Introduce some NaN values artificially (for testing missing value handling)
dataframe.loc[5:10, "plas"] = np.nan  # Simulating missing values

# Split into input (X) and output (Y) variables
X = dataframe.iloc[:, 0:8]  # Features
Y = dataframe.iloc[:, 8]  # Target variable

# Define custom function to drop a column & create new feature
def feature_engineering(X):
    """Drop 'test' column and create a new interaction feature 'BMI_Age_Interaction'."""
    X = pd.DataFrame(X, columns=numerical_features + categorical_features)
    
    # Drop the 'test' column if it exists
    if 'test' in X.columns:
        X = X.drop(columns=['test'], errors='ignore')

    # Create a new feature: BMI_Age_Interaction = mass * age
    X["BMI_Age_Interaction"] = X["mass"] * X["age"]

    return X

# Apply FunctionTransformer to apply the feature engineering function
feature_transformer = FunctionTransformer(feature_engineering)

## 1️⃣ Numerical Processing: Handle NaN & Apply Z-Normalization
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Replace NaN with median
    ('scaler', StandardScaler())  # Apply Z-normalization
])

## 2️⃣ Categorical Processing: One-Hot Encoding
categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # Convert categorical to numeric
])

# Combine feature engineering, numerical processing, and categorical encoding in one pipeline
preprocessing_pipeline = Pipeline([
    ('feature_engineering', feature_transformer),  # Custom feature engineering
    ('preprocessor', ColumnTransformer([
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]))
])

# Apply preprocessing transformations
X_transformed = preprocessing_pipeline.fit_transform(X)

# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X_transformed, Y, test_size=0.33, random_state=7)

# Train **XGBoost** Model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")
xgb_model.fit(X_train, Y_train)

# Save **Preprocessing + Feature Engineering Together**
with open("preprocessor.pkl", "wb") as file:
    pickle.dump(preprocessing_pipeline, file)

# Save **XGBoost Model** as Pickle
with open("xgb_model.pkl", "wb") as file:
    pickle.dump(xgb_model, file)  # Saving XGBClassifier using Pickle

print("Preprocessing pipeline (feature engineering + transformations) and XGBoost model saved successfully!")


Preprocessing pipeline (feature engineering + transformations) and XGBoost model saved successfully!


/Users/u45428/opt/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:39:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


#### 2.2 You now should apply the freeze command to get the requirements

#### 2.3 Now install the requirements, load the model and apply it

In [ ]:
import pickle
from feature_engineering import feature_engineering # new code

# Load the preprocessing pipeline (including feature engineering)
with open("preprocessor.pkl", "rb") as file:
    loaded_preprocessor = pickle.load(file)

# Load the XGBoost model (pickled)
with open("xgb_model.pkl", "rb") as file:
    xgb_loaded_model = pickle.load(file)

print("Preprocessing pipeline (feature engineering + transformations) and XGBoost model loaded successfully!")


# Example: New unseen data (single instance)
new_data = pd.DataFrame({
    'preg': [3],    # Categorical feature
    'plas': [150],  # Numerical feature
    'pres': [72],
    'skin': [35],
    'test': [0],    # This will be dropped automatically
    'mass': [33.6],
    'pedi': [0.627],
    'age': [50]
})

# Apply feature engineering and preprocessing together
new_data_transformed = loaded_preprocessor.transform(new_data)

# Make prediction using the loaded XGBoost model
xgb_prediction = xgb_loaded_model.predict(new_data_transformed)

print(f"XGBoost Prediction: {xgb_prediction[0]}")


Preprocessing pipeline (feature engineering + transformations) and XGBoost model loaded successfully!
XGBoost Prediction: 1
